# 01. Classic RAG with a local model

Retrieval-augmented generation in five steps: read the documents, turn them into
vectors with an embedding model, store the vectors, find the closest ones for a
question, and let a chat model answer using only those passages.

Everything runs against LM Studio on your own machine. Run the cells from top to
bottom. Cells whose heading says "requires LM Studio" call the local server; the
other cells work offline.

In [ ]:
%pip install -q openai==2.53.0 chromadb==1.5.9

## 1. Constants

Every setting for this notebook lives here. Change a model name or the base URL
and rerun the notebook. Set `RUN_LM_STUDIO_DEMO` to `False` if you want to read
the notebook without a running server.

In [ ]:
from pathlib import Path

LM_STUDIO_BASE_URL = "http://127.0.0.1:1234/v1"
LM_STUDIO_API_KEY = "lm-studio"
CHAT_MODEL = "qwen/qwen3.5-9b"
EMBEDDING_MODEL = "text-embedding-qwen3-embedding-4b"
TOP_K = 3
DOCUMENTS_DIR = Path("documents")
RUN_LM_STUDIO_DEMO = True  # set to False to run only the offline cells

print("Chat model:", CHAT_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
print("Documents directory:", DOCUMENTS_DIR.resolve())
print("LM Studio cells enabled:", RUN_LM_STUDIO_DEMO)

## 2. Read the corpus (offline)

Every Markdown file under `documents/` becomes one document. Look at the printed
list: those file names are what the model will cite later.

In [ ]:
document_paths = sorted(DOCUMENTS_DIR.rglob("*.md"))
if not document_paths:
    raise FileNotFoundError(f"No Markdown files under {DOCUMENTS_DIR.resolve()}")

documents = [
    (path.relative_to(DOCUMENTS_DIR).as_posix(), path.read_text(encoding="utf-8"))
    for path in document_paths
]

print(f"{len(documents)} documents:")
for source, text in documents:
    print(f"  {source}: {len(text)} characters")

## 3. Connect and check the models (requires LM Studio)

LM Studio speaks the OpenAI API, so the official `openai` client works once you
point `base_url` at the local server. The output lists the models LM Studio has
loaded; both names from the constants cell must appear.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)

if RUN_LM_STUDIO_DEMO:
    available_models = sorted(model.id for model in client.models.list().data)
    print("Available models:", available_models)
    for required_model in (CHAT_MODEL, EMBEDDING_MODEL):
        if required_model not in available_models:
            raise RuntimeError(f"Load {required_model!r} in LM Studio before continuing.")
else:
    print("Skipped. Set RUN_LM_STUDIO_DEMO = True to reach the local server.")

## 4. Turn text into vectors (requires LM Studio)

An embedding model maps a piece of text to a list of numbers. Texts with similar
meaning end up close to each other, which is what makes semantic search work.
The output shows how many numbers one vector has and the first few of them.

In [ ]:
if RUN_LM_STUDIO_DEMO:
    sample = client.embeddings.create(model=EMBEDDING_MODEL, input=["What is RAG?"])
    vector = sample.data[0].embedding
    print("Vector length:", len(vector))
    print("First five values:", [round(value, 4) for value in vector[:5]])
else:
    print("Skipped embedding sample.")

## 5. Index the documents in Chroma (requires LM Studio)

Chroma is the vector store. It calls the same embedding function for the stored
documents and for every later question, so both live in the same vector space.
`EphemeralClient` keeps everything in memory, so the index disappears with the
kernel. The output is the number of indexed documents.

In [ ]:
import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings


class LMStudioEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, client: OpenAI, model: str) -> None:
        self.client = client
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        response = self.client.embeddings.create(model=self.model, input=list(input))
        return [item.embedding for item in response.data]


collection = None
if RUN_LM_STUDIO_DEMO:
    chroma_client = chromadb.EphemeralClient()
    collection = chroma_client.get_or_create_collection(
        name="notebook_rag",
        embedding_function=LMStudioEmbeddingFunction(client, EMBEDDING_MODEL),
    )
    collection.add(
        ids=[f"doc-{index}" for index in range(len(documents))],
        documents=[text for _, text in documents],
        metadatas=[{"source": source} for source, _ in documents],
    )
    print(f"Indexed {collection.count()} documents.")
else:
    print("Skipped indexing.")

## 6. Retrieve the closest passages (requires LM Studio)

The question is embedded with the same model and compared against the stored
vectors. Look at the distances in the output: a smaller distance means a closer
match, and this ranking is the whole of "retrieval".

In [ ]:
QUESTION = "Which models does this demo use and what is retrieval-augmented generation?"

hits = []
if RUN_LM_STUDIO_DEMO:
    result = collection.query(
        query_texts=[QUESTION], n_results=min(TOP_K, collection.count())
    )
    hits = [
        {"text": text, "source": metadata["source"], "distance": distance}
        for text, metadata, distance in zip(
            result["documents"][0], result["metadatas"][0], result["distances"][0]
        )
    ]
    for rank, hit in enumerate(hits, 1):
        preview = hit["text"].replace("\n", " ")[:120]
        print(f"{rank}. {hit['source']} | distance={hit['distance']:.4f} | {preview}...")
else:
    print("Skipped retrieval.")

## 7. Build the grounded prompt (offline)

This is the "augmented" part. The retrieved passages are pasted into the prompt
with their file names, and the system message forbids answering from anything
else. The output previews the prompt the model will receive.

In [ ]:
def grounded_messages(question: str, retrieved_hits: list[dict]) -> list[dict]:
    context = "\n\n".join(
        f"[source: {hit['source']}]\n{hit['text']}" for hit in retrieved_hits
    )
    return [
        {
            "role": "system",
            "content": (
                "Answer only from the supplied context. If the context does not "
                "contain the answer, say that you do not know. Cite the source "
                "file names you used."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


messages = grounded_messages(QUESTION, hits) if hits else []
print(messages[1]["content"][:800] if messages else "The prompt needs retrieved passages first.")

## 8. Generate the answer with citations (requires LM Studio)

The model now sees only the retrieved passages. Check the answer against the file
names it cites: if a claim has no citation, it did not come from the corpus.

In [ ]:
if RUN_LM_STUDIO_DEMO:
    response = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, temperature=0.2
    )
    print(response.choices[0].message.content or "")
else:
    print("Skipped generation. Set RUN_LM_STUDIO_DEMO = True and rerun from cell 3.")